# Model & Quality Monitors Plus Dashboard & Reports

This code adds Data & Model Quality Monitors in the AWS environment to track shifts or degredation in the data or model. The code here follows the following steps: 

1. Environment Setup
2. Endpoint Deployment with Data Capture Enabled
3. Data Monitor Setup, plus Monitoring Schedule
4. Model Monitor Setup, plus Monitoring Schedule
5. Executing Models to see what production data will be like
6. Infrastructure Monitors and Alarms
7. Dashboard
8. Verification code, to ensure monitors are workings as expected
9. Monitor Report Download
10. Cleanup

In a production environment, where data is not static, it is critical to monitor different systems in order to ensure proper functionality. By monitoring the infrastructure, as well as the data, and the model itself, we are able to track performance and alert the appropriate team if anything isn't operating as intended. 

Attribution: This code was made with the help of AWS tutorials, reference of lab resources in AAI 540, as well as both Claude Code and Perplexity accessed February, 2026.

## 1. Environment Setup

In [1]:
#1.1 Library Imports
import sagemaker
from sagemaker import Session
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker import image_uris, get_execution_role
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DatasetFormat,
    CronExpressionGenerator,
    EndpointInput,
)

import s3fs
import boto3
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timedelta, timezone
from io import StringIO
from pathlib import Path

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [6]:
#1.2 Configuration

region = "us-east-1"
sm_client = boto3.client("sagemaker", region_name=region)
cw_client = boto3.client("cloudwatch", region_name=region)
s3_client = boto3.client("s3", region_name=region)

role = sagemaker.get_execution_role()
region = "us-east-1"
session = sagemaker.Session(boto_session=boto3.Session(region_name=region)) 
bucket = session.default_bucket()  # Dynamically fetches/creates
prefix = "models/benchmarks"

## 2. Deploy Endpoint with Data Capture for Model Monitor

In [11]:
#2.1 Locate model artifacts
local_base = Path("/tmp/Models/benchmarks")
s3_base = f"s3://{bucket}/models/benchmarks"

xgb_paths = {
    "local_tar.gz": local_base / "xgboost/model.tar.gz",
    "s3_tar.gz": f"{s3_base}/xgboost/model.tar.gz",
}

# For loading/testing locally
local_model_path = xgb_paths["local_tar.gz"]
if local_model_path.exists():
    print("Local model exists for testing")
else:
    print("No local; using S3")

# ALWAYS use S3 for model_data in Model/deploy
model_data_uri = xgb_paths["s3_tar.gz"]

# Auto-upload local to S3 if needed
if local_model_path.exists():
    s3_client = boto3.client('s3', region_name=region)
    model_key = model_data_uri.split('/', 3)[-1]  # Extract key
    s3_client.upload_file(str(local_model_path), bucket, model_key)
    print(f"Uploaded local to {model_data_uri}")

Local model exists for testing
Uploaded local to s3://sagemaker-us-east-1-513691803389/models/benchmarks/xgboost/model.tar.gz


In [13]:
#2.2 Container, pre-built Docker image for AWS that runs XGBoost for training, batch transform, or inference
xgboost_container = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
)

print(f"XGBoost image: {xgboost_container}")

XGBoost image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


In [16]:
#2.3 Model & Endpoint
xg_model = sagemaker.Model(
    image_uri=xgboost_container,  
    model_data=model_data_uri,    
    role=role,
    sagemaker_session=session
)

xgb_endpoint_name = f"xgb-benchmark-endpoint-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

data_capture_prefix = f"{prefix}/datacapture"
data_capture_s3_uri = f"s3://{bucket}/{data_capture_prefix}"

# Deploy with data capture enabled
xg_predictor = xg_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=xgb_endpoint_name,
    data_capture_config=sagemaker.model_monitor.DataCaptureConfig(
        enable_capture=True,
        sampling_percentage=100,
        destination_s3_uri=data_capture_s3_uri,
        capture_options=["REQUEST", "RESPONSE"],
    ),
)

print("Endpoint:", xgb_endpoint_name)

------!Endpoint: xgb-benchmark-endpoint-20260212-060251


In [17]:
#2.4 Wait until endpoint is InService
print("Waiting for endpoint to be ready...")
while True:
    resp = sm_client.describe_endpoint(EndpointName=xgb_endpoint_name)
    status = resp["EndpointStatus"]
    print(" Status:", status)
    if status == "InService":
        print("Endpoint is ready!")
        break
    if status == "Failed":
        raise RuntimeError(f"Endpoint deployment failed: {resp.get('FailureReason')}")
    time.sleep(30)

Waiting for endpoint to be ready...
 Status: InService
Endpoint is ready!


## 3. DQ Data Monitor
Data Quality

In [20]:
#3.0 Dynamic URI Finder
bucket = sagemaker.Session().default_bucket()  # Confirms your bucket
prefix = "models/benchmarks"
baseline_key = f"{prefix}/baseline_normalized.csv"
baseline_dataset_uri = f"s3://{bucket}/{baseline_key}"

# Verify exists
s3_client = boto3.client('s3')
s3_client.head_object(Bucket=bucket, Key=baseline_key)
print(f"✅ Dataset: {baseline_dataset_uri}")


✅ Dataset: s3://sagemaker-us-east-1-513691803389/models/benchmarks/baseline_normalized.csv


In [18]:
#3.1: Create Data Quality Monitor
dq_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session,
)

In [21]:
#3.2: Generate baseline using your data (creates stats/constraints)
dq_baseline_uri = f"s3://{bucket}/{prefix}/monitoring/dq-baseline"

# Run baselining job
dq_baseline_job = dq_monitor.suggest_baseline(
    baseline_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=dq_baseline_uri,
    wait=True,
    logs=False,
)

print("✅ DQ baseline created!")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-12-06-10-29-098


...........................................................!✅ DQ baseline created!


In [22]:
#3.3: Get the generated stats/constraints URIs
try:
    dq_stats_uri = dq_monitor.latest_baselining_job.baseline_statistics.file_name
    dq_constraints_uri = dq_monitor.latest_baselining_job.suggested_constraints.file_name
except:
    # Fallback: find files manually
    s3 = boto3.client("s3")
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/dq-baseline/")
    for obj in resp.get("Contents", []):
        if "statistics.json" in obj["Key"]:
            dq_stats_uri = f"s3://{bucket}/{obj['Key']}"
        if "constraints.json" in obj["Key"]:
            dq_constraints_uri = f"s3://{bucket}/{obj['Key']}"

print("Data Quality Stats:", dq_stats_uri)
print("Data Quality Constraints:", dq_constraints_uri)

Data Quality Stats: s3://sagemaker-us-east-1-513691803389/models/benchmarks/monitoring/dq-baseline/statistics.json
Data Quality Constraints: s3://sagemaker-us-east-1-513691803389/models/benchmarks/monitoring/dq-baseline/constraints.json


In [23]:
# 3.4: Create monitoring schedule using new baseline
schedule_name_xgb_dq = "xgb-data-quality-schedule"

try:
    dq_monitor.delete_monitoring_schedule(schedule_name_xgb_dq)
except:
    pass  # No existing schedule

dq_monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name_xgb_dq,
    endpoint_input=xgb_endpoint_name,
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/data-quality/xgb",
    statistics=dq_stats_uri,
    constraints=dq_constraints_uri,
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print("✅ Data quality schedule live:", schedule_name_xgb_dq)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-data-quality-schedule


✅ Data quality schedule live: xgb-data-quality-schedule


In [24]:
dq_monitor.describe_schedule()

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:513691803389:monitoring-schedule/xgb-data-quality-schedule',
 'MonitoringScheduleName': 'xgb-data-quality-schedule',
 'MonitoringScheduleStatus': 'Pending',
 'MonitoringType': 'DataQuality',
 'CreationTime': datetime.datetime(2026, 2, 12, 6, 15, 32, 565000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 2, 12, 6, 15, 32, 640000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)'},
  'MonitoringJobDefinitionName': 'data-quality-job-definition-2026-02-12-06-15-31-598',
  'MonitoringType': 'DataQuality'},
 'EndpointName': 'xgb-benchmark-endpoint-20260212-060251',
 'ResponseMetadata': {'RequestId': 'e1349997-37ba-4230-b74e-ab74f30bbeb4',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'e1349997-37ba-4230-b74e-ab74f30bbeb4',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-po

In [25]:
dq_executions = dq_monitor.list_executions()
dq_executions

[]

## 4. MQ Model Monitor 
Model Quality

In [26]:
#4.1 Create Model Quality Monitor
mq_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [27]:
#4.2 Create a model quality baseline dataset by hitting the endpoint

# Load data
baseline_df = pd.read_csv(f"s3://{bucket}/{prefix}/baseline_normalized.csv")
X_baseline = baseline_df.drop("target", axis=1)

# Get predictions from endpoint
csv_payload = X_baseline.to_csv(header=False, index=False)
predictor = Predictor(endpoint_name=xgb_endpoint_name, sagemaker_session=session)
response = predictor.predict(
    data=csv_payload,
    initial_args={"ContentType": "text/csv", "Accept": "text/csv"},
)

# Parse multi-class predictions
predictions_text = response.decode("utf-8").strip().split("\n")
prob_matrix = np.array([[float(x) for x in line.split(",") if x] for line in predictions_text])
pred_labels = np.argmax(prob_matrix, axis=1)  # argmax for multiclass

# Create baseline dataset
mq_baseline_df = pd.DataFrame({
    'prediction': pred_labels,
    'ground_truth_label': baseline_df['target'].values
})

mq_baseline_key = f"{prefix}/mq_baseline.csv"
csv_buffer = StringIO()
mq_baseline_df.to_csv(csv_buffer, index=False)
s3_client.put_object(
    Bucket=bucket,
    Key=mq_baseline_key,
    Body=csv_buffer.getvalue(),
    ContentType="text/csv",
)

mq_baseline_uri = f"s3://{bucket}/{mq_baseline_key}"
print("✅ MQ baseline uploaded:", mq_baseline_uri)

✅ MQ baseline uploaded: s3://sagemaker-us-east-1-513691803389/models/benchmarks/mq_baseline.csv


In [28]:
#4.2 Run baselining from endpoint deployed model predictions locally
mq_baseline_folder = f"s3://{bucket}/{prefix}/mq-baseline"

mq_monitor.suggest_baseline(
    baseline_dataset=mq_baseline_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/mq-baseline-simple",
    problem_type="MulticlassClassification",
    inference_attribute="prediction",
    ground_truth_attribute="ground_truth_label",
    wait=True,
    logs=False,
)

print("✅ Baselining job complete!")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-12-06-15-51-357


...........................................................!✅ Baselining job complete!


In [29]:
#4.3 Get baseline files
job_desc = mq_monitor.latest_baselining_job.describe()
output_uri = job_desc['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']
mq_stats_uri = f"{output_uri}/statistics.json"
mq_constraints_uri = f"{output_uri}/constraints.json"
print("✅ Stats:", mq_stats_uri)
print("✅ Constraints:", mq_constraints_uri)

✅ Stats: s3://sagemaker-us-east-1-513691803389/models/benchmarks/monitoring/mq-baseline-simple/statistics.json
✅ Constraints: s3://sagemaker-us-east-1-513691803389/models/benchmarks/monitoring/mq-baseline-simple/constraints.json


In [36]:
baseline_job = dq_monitor.latest_baselining_job

#Stats: Extract numerical means/stds
stats_list = []
for feature in baseline_job.baseline_statistics().body_dict["features"]:
    name = feature['name']
    num_stats = feature.get('numerical_statistics', {})
    common = num_stats.get('common', {})
    stats_list.append({
        'Feature': name,
        'Type': feature['inferred_type'],
        'Mean': round(num_stats.get('mean', 0), 2),
        'Std': round(num_stats.get('std_dev', 0), 2),
        'Min': round(num_stats.get('min', 0), 2),
        'Max': round(num_stats.get('max', 0), 2),
        'Count': common.get('num_present', 0)
    })
stats_df = pd.DataFrame(stats_list)
print("📊 Fixed DQ Stats:")
print(stats_df.to_markdown(index=False, numalign="right", stralign="left"))


#Constraints: Key rules
constraints_list = []
for feature in baseline_job.suggested_constraints().body_dict["features"]:
    name = feature['name']
    constraints_list.append({
        'Feature': name,
        'Completeness': feature['completeness'],
        'Constraints': str(feature.get('num_constraints', {}))
    })
constraints_df = pd.DataFrame(constraints_list)
print("\n📏 DQ Constraints:")
print(constraints_df.to_markdown(index=False, numalign="right", stralign="left"))

📊 Fixed DQ Stats:
| Feature   | Type       |    Mean |     Std |      Min |     Max |   Count |
|:----------|:-----------|--------:|--------:|---------:|--------:|--------:|
| meanfreq  | Fractional | 2348.25 | 1468.42 |        0 | 7655.34 |   10242 |
| sd        | Fractional | 2447.62 | 1105.57 |        0 | 6324.35 |   10242 |
| median    | Fractional | 4593.04 | 2696.54 |        0 | 14629.6 |   10242 |
| q25       | Fractional |    0.09 |    0.05 |        0 |    0.32 |   10242 |
| q75       | Fractional | -404.31 |  106.93 | -1131.37 |    0.25 |   10242 |
| iqr       | Fractional |   87.61 |   25.56 |    -4.72 |  169.43 |   10242 |
| skew      | Fractional |   19.79 |    22.2 |   -53.82 |    61.2 |   10242 |
| kurt      | Fractional |   17.91 |   11.75 |   -32.02 |   52.93 |   10242 |
| sp_ent    | Fractional |    3.57 |   10.36 |   -36.42 |   42.49 |   10242 |
| sfm       | Fractional |    1.53 |    7.32 |    -24.4 |   26.74 |   10242 |
| mode      | Fractional |   -4.15 |    6.85 |

In [38]:
#4.4 Create a Model Quality Monitoring Schedule

mq_schedule_name = "xgb-model-quality-schedule"

mq_monitor.create_monitoring_schedule(
    monitor_schedule_name=mq_schedule_name,
    endpoint_input=EndpointInput(
        endpoint_name=xgb_endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="0",
    ),
    problem_type="MulticlassClassification",
    ground_truth_input=f"s3://{bucket}/{prefix}/ground-truth",
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/model-quality",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print("Model quality schedule:", mq_schedule_name)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-model-quality-schedule


Model quality schedule: xgb-model-quality-schedule


In [100]:
#4.5 Create a Ground Truth File for Current Hour (need to run for each new hour)

s3 = boto3.client('s3')
target_hour = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
hour_folder = target_hour.strftime('%Y/%m/%d/%H')

gt_key = f"{xgb_endpoint_name}/AllTraffic/{hour_folder}/groundtruth.jsonl"

records = [json.dumps({
    "groundTruthData": {"data": str(int(label)), "encoding": "CSV"},
    "eventMetadata": {"eventId": f"gt-{i}"},
    "eventVersion": "0"
}) for i, label in enumerate(baseline_df['target'].iloc[:300])]

s3.put_object(
    Bucket=bucket, Key=gt_key, Body='\n'.join(records),
    ContentType='application/jsonlines'
)
print(f"✅ GT: s3://{bucket}/{gt_key}")

✅ GT: s3://sagemaker-us-east-1-513691803389/xgb-benchmark-endpoint-20260212-060251/AllTraffic/2026/02/12/07/groundtruth.jsonl


In [101]:
#4.6 Send predictions
test_data = baseline_df.drop("target", axis=1).iloc[:100]
predictor.predict(test_data.to_csv(header=False, index=False), 
                  initial_args={'ContentType': 'text/csv'})
print("✅ Predictions + ground truth ready!")

✅ Predictions + ground truth ready!


In [79]:
#4.8 Check monitor and executions
mq_monitor.describe_schedule()

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:513691803389:monitoring-schedule/xgb-model-quality-schedule',
 'MonitoringScheduleName': 'xgb-model-quality-schedule',
 'MonitoringScheduleStatus': 'Scheduled',
 'MonitoringType': 'ModelQuality',
 'CreationTime': datetime.datetime(2026, 2, 12, 6, 27, 39, 272000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 2, 12, 7, 15, 38, 285000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)'},
  'MonitoringJobDefinitionName': 'model-quality-job-definition-2026-02-12-06-27-38-569',
  'MonitoringType': 'ModelQuality'},
 'EndpointName': 'xgb-benchmark-endpoint-20260212-060251',
 'LastMonitoringExecutionSummary': {'MonitoringScheduleName': 'xgb-model-quality-schedule',
  'ScheduledTime': datetime.datetime(2026, 2, 12, 7, 0, tzinfo=tzlocal()),
  'CreationTime': datetime.datetime(2026, 2, 12, 7, 5, 18, 143000, tzinfo=tzlocal()),
  'LastModifiedTime': datetime.datetime(20

In [80]:
mq_executions = mq_monitor.list_executions()
mq_executions

## 5. Trigger Executions Manually

What happens during runs: 

1. DataCapture/*.jsonl    → Monitor input
2. ground-truth/*.jsonl   → GT input  
3. Merge job              → Joins predictions+GT
4. Quality analysis       → Computes F1/accuracy
5. S3 reports + CloudWatch → f1=0.94, auc=0.93 🎉
   

In [74]:
xgb_endpoint_name = new_xgb_endpoint_name

In [73]:
#5.1 Confirm Data Flow
now_hh = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0).strftime('%Y%m%d%H')

print(f"Checking hour {now_hh} UTC...")

dc_path = f"{bucket}/{prefix}/datacapture/{xgb_endpoint_name}/AllTraffic/{now_hh}"
gt_path = f"{bucket}/{prefix}/ground-truth/{now_hh}"

dc_ok = False
gt_ok = False

try:
    dc_files = fs.ls(dc_path, detail=True)
    print(f"✅ DataCapture: {len(dc_files)} files")
    dc_ok = True
except:
    print("⏳ DataCapture: Writing...")

try:
    gt_files = fs.ls(gt_path, detail=True)
    print(f"✅ GroundTruth: {len(gt_files)} files")
    gt_ok = True
except:
    print("❌ GroundTruth: Missing")

if dc_ok and gt_ok:
    print("\n🚀 DATA READY → Run manual trigger!")
else:
    print("\n⏳ Wait 2 mins → re-run")


Checking hour 2026021206 UTC...
⏳ DataCapture: Writing...
✅ GroundTruth: 1 files

⏳ Wait 2 mins → re-run


In [81]:
#5.2 Check Flow and Manually Trigger Monitors

print("🔄 Data Flow Monitor + Manual Trigger\n")

schedules = [
    "xgb-data-quality-schedule",
    "xgb-model-quality-schedule"
]

all_done = False

# Step 1: Trigger monitors
for schedule in schedules:
    try:
        sm_client.start_monitoring_schedule(MonitoringScheduleName=schedule)
        print(f"✅ {schedule} triggered!")
    except Exception as e:
        print(f"⚠️ {schedule}: {e}")

print("\n=== Data Flow + Progress ===")

while True:
    now_utc = datetime.now(timezone.utc)
    # Both use YYYY/MM/DD/HH — this is what the monitor uses internally
    dc_dir = now_utc.strftime('%Y/%m/%d/%H')
    gt_dir = now_utc.strftime('%Y/%m/%d/%H')
    
    # Data Flow Check
    dc_path = f"{bucket}/{prefix}/datacapture/{xgb_endpoint_name}/AllTraffic/{dc_dir}"
    gt_path = f"{bucket}/{prefix}/ground-truth/{gt_dir}"
    
    dc_status = "⏳ Writing..."
    gt_status = "❌ Missing"
    
    try:
        dc_files = fs.ls(dc_path, detail=True)
        dc_status = f"✅ {len(dc_files)} files"
    except:
        pass
    
    try:
        gt_files = fs.ls(gt_path, detail=True)
        gt_status = f"✅ {len(gt_files)} files"
    except:
        pass
    
    # Monitor Status
    print(f"[{now_utc.strftime('%H:%M UTC')}]", end=" ")
    print(f"📥 DataCapture/{dc_dir}: {dc_status:<15} | 📤 GT/{gt_dir}: {gt_status:<12}", end="")
    
    all_done = True
    for schedule in schedules:
        resp = sm_client.describe_monitoring_schedule(MonitoringScheduleName=schedule)
        try:
            status = resp["LastMonitoringExecutionSummary"]["MonitoringExecutionStatus"]
            print(f"| {schedule}: {status:<20}", end="")
            if status not in ["Completed", "CompletedWithViolations", "Failed"]:
                all_done = False
        except KeyError:
            print(f"| {schedule}: No run yet      ", end="")
            all_done = False
    
    print()
    
    if dc_status.startswith("✅") and gt_status.startswith("✅") and all_done:
        print("\n🎉 FULL SUCCESS! Check S3 reports + dashboard!")
        break
        
    time.sleep(90)

print("\n📊 Victory checks:")
# Reports
try:
    reports = fs.ls(f"{bucket}/{prefix}/monitoring/", detail=True)
    print(f"✅ New reports: {len([f for f in reports if f['name'].count('/') > 4])}")
except:
    pass

# CloudWatch metrics
cw = boto3.client('cloudwatch')
resp = cw.get_metric_statistics(
    Namespace="AWS/SageMaker",
    MetricName="InvocationsPerInstance",
    Dimensions=[{"Name": "EndpointName", "Value": xgb_endpoint_name}],
    StartTime=datetime.now(timezone.utc) - timedelta(minutes=30),
    EndTime=datetime.now(timezone.utc),
    Period=300,
    Statistics=["Sum"]
)
print(f"✅ Invocations: {len(resp.get('Datapoints', []))} points")

print("\n🏆 MLOps LIVE! Dashboard: SageMaker-ML-Benchmarks")

🔄 Data Flow Monitor + Manual Trigger

✅ xgb-data-quality-schedule triggered!
✅ xgb-model-quality-schedule triggered!

=== Data Flow + Progress ===
[07:15 UTC] 📥 DataCapture/2026/02/12/07: ⏳ Writing...    | 📤 GT/2026/02/12/07: ✅ 1 files   | xgb-data-quality-schedule: No run yet      | xgb-model-quality-schedule: Failed              


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:69                                                                                   │
│                                                                                                  │
│   66 │   │   print("\n🎉 FULL SUCCESS! Check S3 reports + dashboard!")                           │
│   67 │   │   break                                                                               │
│   68 │                                                                                           │
│ ❱ 69 │   time.sleep(90)                                                                          │
│   70                                                                                             │
│   71 print("\n📊 Victory checks:")                                                               │
│   72 # Reports                                                                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyboardInterrupt

## Additional Troubleshooting

In [103]:
# Check row count
s3_client = boto3.client('s3')
resp = s3_client.get_object(Bucket=bucket, Key=f"{endpoint_name}/AllTraffic/2026/02/12/07/groundtruth.jsonl")
lines = resp['Body'].read().decode().count('\n') + 1
print(f"GT lines: {lines}")  # Needs 100+

# DataCapture count too
dc_count = len([o for o in s3_client.list_objects_v2(Bucket=bucket, Prefix=f"{endpoint_name}/AllTraffic/2026/02/12/07/data-capture-").get('Contents', [])])
print(f"DataCapture files: {dc_count}")  # Needs 50+


GT lines: 5
DataCapture files: 0


In [105]:
sm_client = boto3.client('sagemaker')
endpoint_name = 'xgb-benchmark-endpoint-20260212-060251'
config_name = sm_client.describe_endpoint(EndpointName=xgb_endpoint_name)['EndpointConfigName']

config = sm_client.describe_endpoint_config(EndpointConfigName=config_name)
print("DataCapture:", config['ProductionVariants'][0].get('DataCaptureConfigDict', 'MISSING!'))


DataCapture: MISSING!


In [83]:
sm_client = boto3.client('sagemaker')

# Get latest model quality execution
schedule_desc = sm_client.describe_monitoring_schedule(MonitoringScheduleName='xgb-model-quality-schedule')
latest_exec = schedule_desc['LastMonitoringExecutionSummary']

print(f"Status: {latest_exec['MonitoringExecutionStatus']}")
print(f"Failure Reason: {latest_exec.get('FailureReason', 'None')}")  # Key info!

# Processing job ARN for logs
exec_arn = latest_exec['ProcessingJobArn']
job_desc = sm_client.describe_processing_job(ProcessingJobName=exec_arn.split('/')[-1])
print(f"Exit Message: {job_desc.get('ExitMessage', 'None')}")
print(f"Logs: {job_desc['ProcessingLogsArn']}")


Status: Failed
Failure Reason: Job inputs had no data
Exit Message: None


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:14                                                                                   │
│                                                                                                  │
│   11 exec_arn = latest_exec['ProcessingJobArn']                                                  │
│   12 job_desc = sm_client.describe_processing_job(ProcessingJobName=exec_arn.split('/')[-1])     │
│   13 print(f"Exit Message: {job_desc.get('ExitMessage', 'None')}")                               │
│ ❱ 14 print(f"Logs: {job_desc['ProcessingLogsArn']}")                                             │
│   15                                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyError: 'ProcessingLogsArn'

## 6. Infrastructure Monitors (CloudWatch Alarms)

In [51]:
#Infrastructure monitoring alarms

endpoint_metric_dimensions = [
    {"Name": "EndpointName", "Value": new_xgb_endpoint_name},
]

# Latency alarm
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-High-Latency",
    AlarmDescription="Model latency above 10 seconds",
    Namespace="AWS/SageMaker",
    MetricName="ModelLatency",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Average",
    Period=300,
    EvaluationPeriods=2,
    Threshold=10000.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

# Invocation errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-Invocation-Errors",
    AlarmDescription="Invocation errors for endpoint",
    Namespace="AWS/SageMaker",
    MetricName="ModelInvocationErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=1,
    Threshold=5.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

# 5XX errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-5XX-Errors-High",
    AlarmDescription="5XX error rate high",
    Namespace="AWS/SageMaker",
    MetricName="Model5XXErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=2,
    Threshold=5.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

# 4XX errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-4XX-Errors-High",
    AlarmDescription="4XX error rate high",
    Namespace="AWS/SageMaker",
    MetricName="Model4XXErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=2,
    Threshold=10.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

print("Infrastructure alarms created.")


Infrastructure alarms created.


In [54]:
#6.2 Trigger alarm
from urllib.parse import quote

cw_client = boto3.client('cloudwatch')
region = "us-east-1"  # Your region
account_id = boto3.client('sts').get_caller_identity()['Account']

schedule_name = "xgb-model-quality-schedule"
alarm_name = "XGB-F2-Score-Low"

cw_client.put_metric_alarm(
    AlarmName=alarm_name,
    AlarmDescription="F2 score below baseline (drift detected)",
    ActionsEnabled=False,
    MetricName="f2",
    Namespace="aws/sagemaker/Endpoints/model-metrics",
    Statistic="Average",
    Dimensions=[
        {"Name": "Endpoint", "Value": endpoint_name},
        {"Name": "MonitoringSchedule", "Value": schedule_name}
    ],
    Period=600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=0.625,
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="notBreaching"
)

# Direct AWS Console URL
alarm_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#" + \
            f"alarmsV2:alarm/{quote(alarm_name, safe='')}"
print(f"✅ F2 Score alarm created!")
print(f"🔗 View alarm: {alarm_url}")
print(f"📱 Click: https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#alarmsV2:alarm/{alarm_name.replace('/', '%2F')}")

✅ F2 Score alarm created!
🔗 View alarm: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#alarmsV2:alarm/XGB-F2-Score-Low
📱 Click: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#alarmsV2:alarm/XGB-F2-Score-Low


## 7. CloudWatch Monitoring Dashboard

In [56]:
#7.1 CloudWatch Dashboard for ML endpoint

dashboard_name = "SageMaker-ML-Benchmarks"

dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Endpoint – Invocations & Latency",
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", new_xgb_endpoint_name],
                    [".", "ModelLatency", ".", "."],
                ],
                "stacked": False,
                "stat": "Average",
                "period": 60,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Endpoint – Errors",
                "metrics": [
                    ["AWS/SageMaker", "ModelInvocationErrors", "EndpointName", new_xgb_endpoint_name],
                    [".", "Invocation4XXErrors", ".", "."],
                    [".", "Invocation5XXErrors", ".", "."],
                ],
                "stacked": False,
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Data Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "DataQualityViolation",
                        "MonitoringSchedule",
                        schedule_name_xgb_dq,
                    ]
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Model Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "ModelQualityViolation",
                        "MonitoringSchedule",
                        "xgb-model-quality-schedule",
                    ]
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 12,
            "width": 24,
            "height": 6,
            "properties": {
                "title": "Alarm States",
                "metrics": [
                    ["AWS/CloudWatch", "AlarmState", "AlarmName", "XGB-Endpoint-High-Latency"],
                    ["...", "XGB-Endpoint-Invocation-Errors"],
                    ["...", "XGB-Endpoint-5XX-Errors-High"],
                    ["...", "XGB-Endpoint-4XX-Errors-High"],
                ],
                "stat": "Maximum",
                "period": 300,
                "region": region,
            },
        },
    ]
}

cw_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body),
)

print("Dashboard created:", dashboard_name)
print(
    f"URL: https://console.aws.amazon.com/cloudwatch/home?region={region}"
    f"#dashboards:name={dashboard_name}"
)


Dashboard created: SageMaker-ML-Benchmarks
URL: https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=SageMaker-ML-Benchmarks


In [64]:
dashboard_name = "SageMaker-ML-Benchmarks-2"

# Enhanced dashboard (your code + alarms + quality metrics)
dashboard_body = {
    "widgets": [
        # Your existing widgets...
        {
            "type": "metric",
            "x": 0, "y": 18, "width": 12, "height": 6,
            "properties": {
                "title": "F2 Score (Model Drift)",
                "metrics": [
                    ["aws/sagemaker/Endpoints/model-metrics", "f2", "Endpoint", endpoint_name, "MonitoringSchedule", "xgb-model-quality-schedule"]
                ],
                "view": "timeSeries",
                "thresholds": [{"color": "#D13212", "value": 0.625, "label": "Alarm Threshold"}],
                "region": region,
                "period": 300
            }
        },
        {
            "type": "metric",
            "x": 12, "y": 18, "width": 12, "height": 6,
            "properties": {
                "title": "Alarm States (1=ALARM)",
                "metrics": [
                    ["AWS/CloudWatch", "AlarmState", "AlarmName", "XGB-F2-Score-Low"],
                    ["...", "XGB-Endpoint-High-Latency"]
                ],
                "view": "timeSeries",
                "yAxis": {"left": {"min": 0, "max": 1}},
                "region": region
            }
        }
    ]
}

cw_client.put_dashboard(DashboardName=dashboard_name, DashboardBody=json.dumps(dashboard_body))

print("🚀 Sending traffic to populate dashboard...")
predictor = Predictor(endpoint_name=new_xgb_endpoint_name, sagemaker_session=session)

# Fixed CW query (no deprecation)
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(minutes=5)

resp = cw_client.get_metric_statistics(
    Namespace="AWS/SageMaker",
    MetricName="Invocations",
    Dimensions=[{"Name": "EndpointName", "Value": new_xgb_endpoint_name}],
    StartTime=start_time,
    EndTime=end_time,
    Period=60,
    Statistics=["Sum"]
)

datapoints = resp.get('Datapoints', [])
total_invocations = sum(p.get('Sum', 0) or 0 for p in datapoints)
print(f"📈 Invocations (last 5 min): {total_invocations} ({len(datapoints)} points)")

if total_invocations > 0:
    print("✅ LIVE metrics flowing!")
else:
    print("⏳ Wait 1-2 min → refresh")

# Dashboard (your existing)
dashboard_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={quote(dashboard_name)}"
print(f"📊 LIVE: {dashboard_url}")

🚀 Sending traffic to populate dashboard...
📈 Invocations (last 5 min): 0 (0 points)
⏳ Wait 1-2 min → refresh
📊 LIVE: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=SageMaker-ML-Benchmarks-2


## 8. Verification

In [68]:
#Verify data capture and ground truth ready

fs = s3fs.S3FileSystem()
bucket = sagemaker.Session().default_bucket() 
prefix = "models/benchmarks"
endpoint_name = new_xgb_endpoint_name 
now_utc = datetime.now(timezone.utc)
current_hour = now_utc.strftime('%Y/%m/%d/%H')

print(f"✅ Using bucket: {bucket}")
print(f"⏰ Hour: {current_hour}")

dc_path = f"{bucket}/{prefix}/datacapture/{endpoint_name}/AllTraffic/{current_hour}"
try:
    dc_files = fs.ls(dc_path, detail=True)
    print(f"📥 DataCapture/{current_hour}: ✅ {len(dc_files)} files")
except:
    print(f"📥 DataCapture/{current_hour}: ⏳ Writing...")

# Ground Truth (MQ needs this)
gt_path = f"{bucket}/{prefix}/ground-truth/{current_hour}"
try:
    gt_files = fs.ls(gt_path, detail=True)
    print(f"📤 GT/{current_hour}: ✅ {len(gt_files)} files")
except:
    print(f"📤 GT/{current_hour}: ❌ Missing → re-upload")

print(f"\nMonitor looks for: {dc_path}, {gt_path}")

✅ Using bucket: sagemaker-us-east-1-513691803389
⏰ Hour: 2026/02/12/06
📥 DataCapture/2026/02/12/06: ✅ 3 files
📤 GT/2026/02/12/06: ❌ Missing → re-upload

Monitor looks for: sagemaker-us-east-1-513691803389/models/benchmarks/datacapture/xgb-benchmark-endpoint-20260212-060251/AllTraffic/2026/02/12/06, sagemaker-us-east-1-513691803389/models/benchmarks/ground-truth/2026/02/12/06


In [66]:
def check_monitor_status():
    schedules = [schedule_name_xgb_dq, "xgb-model-quality-schedule"]
    print("=== Monitor Status ===")
    for sched in schedules:
        try:
            resp = sm_client.describe_monitoring_schedule(MonitoringScheduleName=sched)
            exec_summary = resp.get("LastMonitoringExecutionSummary", {})
            status = exec_summary.get("MonitoringExecutionStatus", "No execution")
            time = exec_summary.get("ScheduledTime", "N/A")
            print(f"{sched:25}: {status} ({time})")
        except Exception as e:
            print(f"{sched:25}: Error - {e}")
    print("====================")

check_monitor_status()

=== Monitor Status ===
xgb-data-quality-schedule: No execution (N/A)
xgb-model-quality-schedule: No execution (N/A)


In [37]:
#Full MQ schedule diagnosis
mq_schedule_name = "xgb-model-quality-schedule"

resp = sm_client.describe_monitoring_schedule(MonitoringScheduleName=mq_schedule_name)
print("Schedule Status:", resp["MonitoringScheduleStatus"])
print("Last Execution:", resp.get("LastMonitoringExecutionSummary", "None"))

# List recent executions
executions = sm_client.list_monitoring_executions(
    MonitoringScheduleName=mq_schedule_name,
    MaxResults=5,
    SortOrder="Descending"
)
print("\nRecent Executions:")
for exec in executions["MonitoringExecutionSummaries"]:
    print(f"  {exec['ScheduledTime']}: {exec['MonitoringExecutionStatus']}")

Schedule Status: Pending
Last Execution: {'MonitoringScheduleName': 'xgb-model-quality-schedule', 'ScheduledTime': datetime.datetime(2026, 2, 11, 7, 0, tzinfo=tzlocal()), 'CreationTime': datetime.datetime(2026, 2, 11, 7, 0, 45, 481000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2026, 2, 11, 7, 6, 58, 248000, tzinfo=tzlocal()), 'MonitoringExecutionStatus': 'Failed', 'ProcessingJobArn': 'arn:aws:sagemaker:us-east-1:418418308994:processing-job/groundtruth-merge-202602110700-421f72f21b6818737d32d66f', 'EndpointName': 'xgb-benchmark-endpoint-20260211-061214', 'FailureReason': 'Job inputs had no data'}

Recent Executions:
  2026-02-11 07:00:00+00:00: Failed
  2026-02-10 06:00:00+00:00: Failed
  2026-02-09 08:00:00+00:00: Failed
  2026-02-09 07:00:00+00:00: Failed
  2026-02-09 06:00:00+00:00: Failed


## 9. Generate Model & Data Reports

This is for after the monitor runs.

In [38]:
#Helper to inspect latest executions and get report locations

def get_latest_execution(schedule_name):
    resp = sm_client.list_monitoring_executions(
        MonitoringScheduleName=schedule_name,
        MaxResults=5,
        SortOrder="Descending",
    )
    if not resp["MonitoringExecutionSummaries"]:
        print("No executions found for", schedule_name)
        return None
    return resp["MonitoringExecutionSummaries"][0]

for name in [schedule_name_xgb_dq, "xgb-model-quality-schedule"]:
    latest = get_latest_execution(name)
    if latest:
        print("\nSchedule:", name)
        print(" Status:", latest["MonitoringExecutionStatus"])
        print(" ScheduledTime:", latest["ScheduledTime"])



Schedule: xgb-data-quality-schedule
 Status: CompletedWithViolations
 ScheduledTime: 2026-02-11 07:00:00+00:00

Schedule: xgb-model-quality-schedule
 Status: Failed
 ScheduledTime: 2026-02-11 07:00:00+00:00


In [39]:
#Example: download latest data quality report artifacts

dq_latest = get_latest_execution(schedule_name_xgb_dq)
if dq_latest and "ProcessingJobArn" in dq_latest:
    job_name = dq_latest["ProcessingJobArn"].split("/")[-1]
    job_desc = sm_client.describe_processing_job(ProcessingJobName=job_name)
    outputs = job_desc["ProcessingOutputConfig"]["Outputs"]
    for out in outputs:
        uri = out["S3Output"]["S3Uri"]
        print("DQ output:", uri)
        # Typically contains /constraints.json and /statistics.json


DQ output: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/data-quality/xgb/xgb-benchmark-endpoint-20260211-061214/xgb-data-quality-schedule/2026/02/11/07


In [40]:
#Example: download latest model quality report artifacts

mq_latest = get_latest_execution("xgb-model-quality-schedule")
if mq_latest and "ProcessingJobArn" in mq_latest:
    job_name = mq_latest["ProcessingJobArn"].split("/")[-1]
    job_desc = sm_client.describe_processing_job(ProcessingJobName=job_name)
    outputs = job_desc["ProcessingOutputConfig"]["Outputs"]
    for out in outputs:
        uri = out["S3Output"]["S3Uri"]
        print("MQ output:", uri)


MQ output: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/model-quality/merge


## 10. Clean Up Resources

In [ ]:
# Delete all monitoring schedules
schedules = [
    "xgb-data-quality-schedule",
    "xgb-model-quality-schedule",
    "xgb-model-quality-schedule",
    "xgb-model-quality-schedule"
]

for schedule in schedules:
    try:
        sm_client.delete_monitoring_schedule(MonitoringScheduleName=schedule)
        print(f"✅ Deleted schedule: {schedule}")
    except Exception as e:
        print(f"⚠️ {schedule}: {e}")

In [ ]:
# Delete endpoint (stops data capture)
try:
    sm_client.delete_endpoint(EndpointName=new_xgb_endpoint_name)
    print(f"✅ Deleting endpoint: {new_xgb_endpoint_name}")
except Exception as e:
    print(f"⚠️ Endpoint delete: {e}")


In [ ]:
import boto3
sm_client = boto3.client('sagemaker', region_name='us-east-1')
cw_client = boto3.client('cloudwatch', region_name='us-east-1')

print("=== CLEANUP STATUS ===")
eps = sm_client.list_endpoints()["Endpoints"]
print("Endpoints:", [e["EndpointName"] for e in eps] or "✅ NONE")

schedules = sm_client.list_monitoring_schedules()["MonitoringScheduleSummaries"]
print("Schedules:", [s["MonitoringScheduleName"] for s in schedules] or "✅ NONE")

alarms = cw_client.describe_alarms(AlarmNamePrefix="XGB-")["MetricAlarms"]
print("XGB Alarms:", [a["AlarmName"] for a in alarms] or "✅ NONE")

print("✅ Ready for restart!")
